# Rock-Paper-Lose - model experiments

Iterate on gesture-classifier models here, then export the winner to int8 TFLite for the C++ app.

**Contract with the C++ `TfliteClassifier` (must hold or the port breaks):**
- Class order is pinned to `['rock','paper','scissors','none']` so indices match `main.cpp` (0=rock, 1=paper, 2=scissors).
- Input is a single NHWC, square, 3-channel tensor. Preprocessing/rescaling is **inside the model**, so C++ feeds raw 0-255 pixels.
- Output ends in **softmax** (so `confidence` is a probability).
- Export full-integer **int8** with **uint8** input/output, sized small for >=30 FPS on the Pi 3.

Expects captured data in `../data/<class>/*.jpg` (pull it from the Pi first).

In [ ]:
import os
# Use Keras 2 (tf-keras) - stable TFLite full-integer quantization path.
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

print("TF", tf.__version__, "| Keras", tf.keras.__version__)

DATA_DIR    = "../data"
CLASS_NAMES = ["rock", "paper", "scissors", "none"]  # order MUST match C++ indices
IMG_SIZE    = 96      # try 64 for more FPS headroom on the Pi 3
BATCH       = 32
SEED        = 1337
AUTOTUNE    = tf.data.AUTOTUNE

In [ ]:
# Load data. class_names pins the label order to match the C++ app.
# NOTE: this is a random split - frames from one capture burst are near
# duplicates, so this val accuracy is optimistic. For an honest number, keep a
# whole capture session (or a person) aside as a separate test set.
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="training", seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH,
    label_mode="categorical", class_names=CLASS_NAMES)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="validation", seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH,
    label_mode="categorical", class_names=CLASS_NAMES)

print("classes:", train_ds.class_names)

In [ ]:
# image_dataset_from_directory stretch-resizes to a square. The camera instead
# letterboxes the 4:3 sensor into a square. If real-world accuracy lags the val
# number, swap the resize for tf.image.resize_with_pad to match the camera.
data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),   # RPS shapes are mirror-safe
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomBrightness(0.20, value_range=(0, 255)),
], name="augment")

train_perf = train_ds.map(lambda x, y: (data_aug(x, training=True), y),
                          num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_perf   = val_ds.prefetch(AUTOTUNE)

In [ ]:
# Candidate A: MobileNetV2 transfer learning (small alpha for the Pi 3).
# Rescaling to [-1,1] is INSIDE the model -> C++ feeds raw 0-255.
def build_mobilenet(alpha=0.35):
    base = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3), alpha=alpha,
        include_top=False, weights="imagenet", pooling="avg")
    base.trainable = False
    inputs = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3))           # raw 0-255
    x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1.0)(inputs)
    x = base(x, training=False)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name=f"mobilenetv2_a{alpha}")

# Candidate B: tiny CNN from scratch (smallest/fastest; prob needs more data).
def build_small_cnn():
    inputs = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3))           # raw 0-255
    x = tf.keras.layers.Rescaling(1.0 / 255.0)(inputs)
    for f in (16, 32, 64):
        x = tf.keras.layers.Conv2D(f, 3, padding="same", activation="relu")(x)
        x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="small_cnn")



In [ ]:
def train(model, epochs=12, lr=1e-3):
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="categorical_crossentropy", metrics=["accuracy"])
    return model.fit(train_perf, validation_data=val_perf, epochs=epochs)

def evaluate(model):
    y_true, y_pred = [], []
    for images, labels in val_perf:
        probs = model.predict(images, verbose=0)
        y_pred += list(np.argmax(probs, axis=1))
        y_true += list(np.argmax(labels.numpy(), axis=1))
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))
    print("confusion matrix (rows=true, cols=pred):")
    print(confusion_matrix(y_true, y_pred))

In [ ]:
def _rep_dataset(n_batches=20):
    # Representative data in the model's input domain: raw 0-255 floats.
    for images, _ in train_ds.take(n_batches):
        for i in range(images.shape[0]):
            yield [tf.cast(images[i:i+1], tf.float32)]

def to_int8_tflite(model, path):
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = _rep_dataset
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.uint8    # C++ feeds raw uint8 pixels
    conv.inference_output_type = tf.uint8
    tfl = conv.convert()
    with open(path, "wb") as f:
        f.write(tfl)
    print(f"wrote {path}: {len(tfl)/1024:.1f} KiB")
    return path

def eval_tflite(path):
    """Accuracy of the quantized model = what actually runs on the Pi."""
    interp = tf.lite.Interpreter(model_path=path)
    interp.allocate_tensors()
    inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
    correct = total = 0
    for images, labels in val_ds:
        for i in range(images.shape[0]):
            x = images[i:i+1]
            x = tf.cast(x, inp["dtype"])     # uint8 0-255
            interp.set_tensor(inp["index"], x)
            interp.invoke()
            pred = int(np.argmax(interp.get_tensor(out["index"])[0]))
            correct += int(pred == int(np.argmax(labels[i].numpy())))
            total += 1
    print(f"int8 TFLite val accuracy: {correct/total:.3f}")

    # Latency proxy ONLY - this machine, not the Pi 3. Measure real FPS on the
    # Pi (the perf scripts). Useful here just to compare candidates relatively.
    import time
    dummy = np.zeros(inp["shape"], dtype=inp["dtype"])
    interp.set_tensor(inp["index"], dummy); interp.invoke()  # warmup
    t0 = time.perf_counter()
    for _ in range(100):
        interp.set_tensor(inp["index"], dummy); interp.invoke()
    print(f"~{(time.perf_counter()-t0)/100*1000:.2f} ms/inference (host, not Pi)")

## Candidate A - MobileNetV2 transfer

In [ ]:
m_a = build_mobilenet(alpha=0.35)
train(m_a, epochs=12)
evaluate(m_a)
to_int8_tflite(m_a, "mobilenet_a035.tflite")
eval_tflite("mobilenet_a035.tflite")

## Candidate B - tiny CNN from scratch

In [ ]:
m_b = build_small_cnn()
train(m_b, epochs=20)
evaluate(m_b)
to_int8_tflite(m_b, "small_cnn.tflite")
eval_tflite("small_cnn.tflite")

## Porting the winner to C++

1. Pick the candidate with the best **int8 TFLite val accuracy** that also fits the FPS budget (verify FPS on the Pi, not here).
2. Copy its `.tflite` to the Pi as `gesture.tflite` and run `rock_paper_lose --model gesture.tflite`.
3. `TfliteClassifier` reads input size and class count from the model, so no C++ change is needed as long as the contract above held.
4. When happy, fold the winning recipe into `scripts/train_and_convert.py` for a reproducible, scripted build (a Moodle deliverable).

Later, the **accessory** model is the same pipeline with two folders (`accessory` / `no_accessory`) and `CLASS_NAMES = ['accessory','no_accessory']`.